# Machine Learning-Based Histogram Boosting Approach for Attack Detection in IoT Networks

Code for the paper:

> M. Nouman, N. Cordeschi, A. Fascista, "Machine Learning-Based Histogram Boosting Approach for Attack Detection in IoT Networks," 16th IFIP Wireless and Mobile Networking Conference (WMNC), Leuven, Belgium, 2025, pp. 53–57. [doi:10.23919/WMNC67099.2025.11299281](https://doi.org/10.23919/WMNC67099.2025.11299281)

The notebook follows the three-phase framework of the paper on the CICIoT2023 dataset:

1. **Phase 1** – preprocessing and mapping of the 33 attack classes into 7 attack categories plus benign
2. **Phase 2** – class balancing with ADASYN followed by random undersampling (RUS)
3. **Phase 3** – Histogram Gradient Boosting (HGB) compared with nine other ML and boosting models

The text outputs are from the original Google Colab run used for the paper. The figures are saved in the `figures/` folder.

## Setup

Download `Merged52.csv` from the [CICIoT2023 dataset](https://www.unb.ca/cic/datasets/iotdataset-2023.html) and place it in `data/`, or change `DATA_PATH`.

In [ ]:
import os
import re
import time

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, roc_auc_score, roc_curve, auc,
                             precision_recall_curve, average_precision_score,
                             confusion_matrix, classification_report)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import AdaBoostClassifier, HistGradientBoostingClassifier
from imblearn.over_sampling import ADASYN
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

DATA_PATH = "data/Merged52.csv"
os.makedirs("figures", exist_ok=True)

## Phase 1: Preprocessing and attack mapping

The 33 CICIoT2023 attack classes are grouped into 7 categories based on shared features (DDoS, DoS, Mirai, Brute Force, Spoofing, Recon, Web-based), plus Benign.

In [ ]:
# Function to map subclasses to corresponding superclasses
def map_to_superclass(subclass):
    subclass = str(subclass)

    if subclass.startswith("DDOS-"):
        return "DDOS"
    elif subclass.startswith("DOS-"):
        return "DOS"
    elif subclass.startswith("MIRAI-"):
        return "Mirai"
    elif re.search(r'.*bruteforce.*', subclass, re.IGNORECASE):
        return "Brute_Force"
    elif re.search(r'.*(MITM-ARPSPOOFING|DNS_SPOOFING).*', subclass, re.IGNORECASE):
        return "Spoofing"
    elif subclass.startswith("RECON-") or subclass.startswith("VULNERABILITYSCAN"):
        return "Recon"
    elif re.search(r'.*(Sqlinjection|Commandinjection|BACKDOOR_MALWARE|UPLOADING_ATTACK|XSS|BROWSERHIJACKING).*', subclass, re.IGNORECASE):
        return "Web-based"
    elif subclass.upper() == "BENIGN":
        return "BENIGN"
    return "Nan"

In [ ]:
# Step 1: Load the dataset
df = pd.read_csv(DATA_PATH)
df

       Header_Length  Protocol Type  Time_To_Live           Rate  \
0               20.0              6         64.00   34306.428922   
1                0.0              1         64.00   45206.984264   
2               20.0              6         64.00   22547.597033   
3               20.0              6         64.00   19676.787390   
4                0.0              1         64.00  157207.796102   
...              ...            ...           ...            ...   
65719           32.0              6         58.40     143.022417   
65720           20.0              6         64.00   24432.364420   
65721           16.0              6         67.60     119.572830   
65722           17.6             17        114.40     196.189870   
65723           19.8              6         63.36   22207.359559   

       fin_flag_number  syn_flag_number  rst_flag_number  psh_flag_number  \
0                  0.0              1.0              0.0              0.0   
1                  0.0       

In [ ]:
# Step 2: Apply the mapping function to the target column
df['mapped_target'] = df['Label'].apply(map_to_superclass)

In [ ]:
# Check which original labels are mapped to 'DDOS'
dos_subclasses = df[df['mapped_target'] == 'DDOS']

# Print the relevant subclasses (target variable for DOS subclasses)
print("Subclasses associated with DDOS:")
print(dos_subclasses['Label'].unique())

Subclasses associated with DDOS:
['DDOS-SYN_FLOOD' 'DDOS-ICMP_FLOOD' 'DDOS-SYNONYMOUSIP_FLOOD'
 'DDOS-RSTFINFLOOD' 'DDOS-TCP_FLOOD' 'DDOS-UDP_FLOOD'
 'DDOS-ICMP_FRAGMENTATION' 'DDOS-PSHACK_FLOOD' 'DDOS-UDP_FRAGMENTATION'
 'DDOS-ACK_FRAGMENTATION' 'DDOS-HTTP_FLOOD' 'DDOS-SLOWLORIS']


In [ ]:
# Step 3: Remove rows with missing or empty labels
df = df.dropna(subset=['Label'])
df = df[df['Label'] != ""]

# Step 4: Handle missing and infinity values in features
X = df.drop(columns=['Label', 'mapped_target'])
X = X.replace([np.inf, -np.inf], np.nan)  # Replace infinities with NaN
X = X.fillna(X.mean())  # Fill NaN values with column means

# Step 5: Encode categorical features (if any) and normalize numeric features
# Identify categorical columns
categorical_columns = X.select_dtypes(include=['object']).columns

# Apply LabelEncoder to categorical features
label_encoder = LabelEncoder()
for col in categorical_columns:
    X[col] = label_encoder.fit_transform(X[col])

# Scale numeric features using StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Step 6: Encode the target labels
y = label_encoder.fit_transform(df['mapped_target'])

Class distribution after mapping (original instances in Table I of the paper):

In [ ]:
# Count the occurrences of each class in the 'mapped_target' column
class_counts = df['mapped_target'].value_counts()

# Create a bar plot for class distribution
plt.figure(figsize=(10, 6))
sns.barplot(x=class_counts.index, y=class_counts.values, palette='viridis')

# Add labels and title
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')

# Display count on top of each bar
for i, count in enumerate(class_counts.values):
    plt.text(i, count + 0.1, str(count), ha='center', va='bottom', fontsize=12)

# Rotate x-axis labels for readability
plt.xticks(rotation=45)
plt.show()

## Phase 2: Class balancing with ADASYN and RUS

ADASYN oversamples the minority classes, then random undersampling gives 47,002 samples per class (Table I of the paper).

In [ ]:
# Step 1: Print the class distribution before resampling
print("Class distribution before resampling:")
print(pd.Series(y).value_counts())

# Step 2: Apply ADASYN with a custom sampling strategy
adasyn = ADASYN(sampling_strategy='auto', random_state=42)
X_res_adasyn, y_res_adasyn = adasyn.fit_resample(X, y)

# Step 3: Print the class distribution after applying ADASYN
print("\nClass distribution after applying ADASYN:")
print(pd.Series(y_res_adasyn).value_counts())  # Corrected this line

# Step 4: Apply Random undersampling to further balance the classes
random_Undersampler = RandomUnderSampler(sampling_strategy='auto', random_state=42)
X, y = random_Undersampler.fit_resample(X_res_adasyn, y_res_adasyn)

# Step 5: Print the class distribution after applying Random Undersampling
print("\nClass distribution after applying Random Undersampling:")
print(pd.Series(y).value_counts())

Class distribution before resampling:
2    47729
3    11120
4     3664
0     1471
5      981
6      715
7       25
1       18
Name: count, dtype: int64

Class distribution after applying ADASYN:
5    47857
0    47733
1    47730
2    47729
7    47724
4    47724
6    47706
3    47002
Name: count, dtype: int64

Class distribution after applying Random Undersampling:
0    47002
1    47002
2    47002
3    47002
4    47002
5    47002
6    47002
7    47002
Name: count, dtype: int64


In [ ]:
# Step 7: Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## Phase 3: Model training and evaluation

70% of the data is used for training and 30% for testing. HGB is compared with SGD, LR, NB, LDA, QDA, AdaBoost, XGBoost, CatBoost and LightGBM (Table II of the paper).

In [ ]:
# Initialize Label Encoder
label_encoder = LabelEncoder()

# Encode target labels
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Binarize labels for multi-class AUC and PR Curve computation
n_classes = len(set(y_train_encoded))
y_test_binarized = label_binarize(y_test_encoded, classes=np.arange(n_classes))


# Correct SGDClassifier initialization
models = {
  #  'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'SGD': SGDClassifier(loss='log_loss', random_state=42),  # Corrected loss to 'log_loss'
    #'Perceptron': Perceptron(random_state=42),
    #'Ridge Classifier': RidgeClassifier(),
    'LR': LogisticRegression(max_iter=500, random_state=42),
    #'Decision Tree': DecisionTreeClassifier(random_state=42),
    'NB': GaussianNB(),
    'LDA': LinearDiscriminantAnalysis(),
    'QDA': QuadraticDiscriminantAnalysis(),
    'AdaB': AdaBoostClassifier(n_estimators=100, random_state=42),
    'XGB': XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    'HGB': HistGradientBoostingClassifier(random_state=42),
    'CatB': CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, verbose=0, random_state=42),
    'LGB': LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    #'GradientBoost': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
    #'Extra Trees Classifier': ExtraTreesClassifier(n_estimators=100, random_state=42),
}


# Store metrics for all models
results = {}

# Initialize plot for AUC-ROC curves
plt.figure(figsize=(10, 8))

# Train and evaluate all models
for model_name, model in models.items():
    print(f"\nTraining {model_name}...")

    # Track execution times
    start_total_time = time.time()
    start_train_time = time.time()

    # Train model
    model.fit(X_train, y_train_encoded)

    end_train_time = time.time()

    # Track prediction time
    start_pred_time = time.time()

    # Make predictions
    y_pred = model.predict(X_test)

    # Get probability predictions for Precision-Recall curve and AUC score
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)
        auc_score = roc_auc_score(y_test_binarized, y_prob, multi_class='ovr')
    else:
        # For models like Perceptron, RidgeClassifier, and SGD without predict_proba, we can't compute AUC.
        y_prob = None
        auc_score = np.nan

    end_pred_time = time.time()

    # Compute evaluation metrics
    accuracy = accuracy_score(y_test_encoded, y_pred)
    precision = precision_score(y_test_encoded, y_pred, average='weighted', zero_division=1)
    recall = recall_score(y_test_encoded, y_pred, average='weighted')
    f1 = f1_score(y_test_encoded, y_pred, average='weighted')
    mcc = matthews_corrcoef(y_test_encoded, y_pred)  # Calculate MCC

    # Confusion matrix
    conf_matrix = confusion_matrix(y_test_encoded, y_pred)

    # Calculate error metrics for each class in the confusion matrix
    fnr_list, fdr_list, fpr_list, for_list = [], [], [], []

    # For each class, calculate the error metrics
    for i in range(n_classes):
        tn, fp, fn, tp = conf_matrix[i, i], np.sum(conf_matrix[i, :]) - conf_matrix[i, i], np.sum(conf_matrix[:, i]) - conf_matrix[i, i], np.sum(conf_matrix)

        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
        fdr = fp / (fp + tp) if (fp + tp) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        for_ = fn / (fn + tn) if (fn + tn) > 0 else 0

        fnr_list.append(fnr)
        fdr_list.append(fdr)
        fpr_list.append(fpr)
        for_list.append(for_)

    # Track total execution time
    end_total_time = time.time()

    # Convert label_encoder.classes_ to a list of strings
    target_names = [str(cls) for cls in label_encoder.classes_]

    # Now use target_names in classification_report
    class_report = classification_report(y_test_encoded, y_pred, target_names=target_names)
    print("\nClassification Report:")
    print(class_report)

    # Store results
    results[model_name] = {
        'Classification Report': class_report,  # Add classification report to results
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'MCC': mcc,  # Add MCC to results
        'AUC Score': auc_score,
        'FNR': np.mean(fnr_list),
        'FDR': np.mean(fdr_list),
        'FPR': np.mean(fpr_list),
        'FoR': np.mean(for_list),
        'Confusion Matrix': conf_matrix,
        'Training Time': end_train_time - start_train_time,
        'Prediction Time': end_pred_time - start_pred_time,
        'Total Execution Time': end_total_time - start_total_time,
        'Probabilities': y_prob,
       }

    print(f"\nEvaluating {model_name}:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  MCC: {mcc:.4f}")  # Print MCC
    print(f"  AUC Score: {auc_score:.4f}")
    print(f"  FNR: {np.mean(fnr_list):.4f}, FDR: {np.mean(fdr_list):.4f}, FPR: {np.mean(fpr_list):.4f}, FoR: {np.mean(for_list):.4f}")
    print(f"  Training Time: {end_train_time - start_train_time:.4f} seconds")
    print(f"  Prediction Time: {end_pred_time - start_pred_time:.4f} seconds")
    print(f"  Total Execution Time: {end_total_time - start_total_time:.4f} seconds")
    print("  Confusion Matrix:")
    print(conf_matrix)
    print('-' * 50)


    # Plot Confusion Matrix with Class Names and Increased Text Size
    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(y_test_encoded, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", annot_kws={"size": 16},
            xticklabels=label_encoder.classes_,  # Use label_encoder.classes_ for x-axis labels
            yticklabels=label_encoder.classes_) # Use label_encoder.classes_ for y-axis labels
    plt.xlabel("Predicted Labels", fontsize=14)
    plt.ylabel("True Labels", fontsize=14)
    plt.title(f"Confusion Matrix for {model_name}", fontsize=16)
    plt.xticks(rotation=45, ha='right', fontsize=12)  # Rotate x-axis labels for better readability
    plt.yticks(fontsize=12)  # Increase y-axis label font size
    plt.tight_layout()  # Adjust layout to prevent overlapping elements
    plt.show()  # Display the confusion matrix


Training SGD...

Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.17      0.22     14210
           1       0.67      0.65      0.66     13933
           2       0.92      0.58      0.71     13992
           3       0.69      0.94      0.80     14169
           4       0.98      0.99      0.98     14141
           5       0.50      0.48      0.49     14014
           6       0.38      0.52      0.44     14121
           7       0.58      0.69      0.63     14225

    accuracy                           0.63    112805
   macro avg       0.63      0.63      0.62    112805
weighted avg       0.63      0.63      0.62    112805


Evaluating SGD:
  Accuracy: 0.6264
  Precision: 0.6304
  Recall: 0.6264
  F1 Score: 0.6157
  MCC: 0.5766
  AUC Score: 0.9219
  FNR: 0.0438, FDR: 0.0438, FPR: 0.3737, FoR: 0.3691
  Training Time: 21.2042 seconds
  Prediction Time: 0.3240 seconds
  Total Execution Time: 21.5977 seconds
  Confusion Matrix:
[[


Training LR...



Classification Report:
              precision    recall  f1-score   support

           0       0.36      0.29      0.32     14210
           1       0.68      0.74      0.71     13933
           2       0.92      0.58      0.71     13992
           3       0.69      0.94      0.80     14169
           4       0.99      1.00      0.99     14141
           5       0.55      0.45      0.50     14014
           6       0.45      0.46      0.45     14121
           7       0.60      0.76      0.67     14225

    accuracy                           0.65    112805
   macro avg       0.66      0.65      0.64    112805
weighted avg       0.65      0.65      0.64    112805


Evaluating LR:
  Accuracy: 0.6525
  Precision: 0.6549
  Recall: 0.6525
  F1 Score: 0.6442
  MCC: 0.6051
  AUC Score: 0.9263
  FNR: 0.0411, FDR: 0.0409, FPR: 0.3476, FoR: 0.3446
  Training Time: 152.7849 seconds
  Prediction Time: 0.3190 seconds
  Total Execution Time: 153.1744 seconds
  Confusion Matrix:
[[ 4154  1726     


Training NB...

Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.02      0.03     14210
           1       0.57      0.60      0.58     13933
           2       0.97      0.45      0.62     13992
           3       0.65      0.93      0.77     14169
           4       0.91      1.00      0.95     14141
           5       0.81      0.05      0.09     14014
           6       0.36      0.02      0.05     14121
           7       0.27      1.00      0.42     14225

    accuracy                           0.51    112805
   macro avg       0.63      0.51      0.44    112805
weighted avg       0.63      0.51      0.44    112805


Evaluating NB:
  Accuracy: 0.5098
  Precision: 0.6286
  Recall: 0.5098
  F1 Score: 0.4389
  MCC: 0.4898
  AUC Score: 0.8804
  FNR: 0.0492, FDR: 0.0555, FPR: 0.4908, FoR: 0.3705
  Training Time: 0.2470 seconds
  Prediction Time: 0.8899 seconds
  Total Execution Time: 1.1899 seconds
  Confusion Matrix:
[[  25


Training LDA...

Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.30      0.31     14210
           1       0.70      0.72      0.71     13933
           2       0.90      0.45      0.60     13992
           3       0.65      0.98      0.78     14169
           4       0.96      0.92      0.94     14141
           5       0.51      0.43      0.47     14014
           6       0.40      0.47      0.43     14121
           7       0.54      0.59      0.56     14225

    accuracy                           0.61    112805
   macro avg       0.62      0.61      0.60    112805
weighted avg       0.62      0.61      0.60    112805


Evaluating LDA:
  Accuracy: 0.6069
  Precision: 0.6231
  Recall: 0.6069
  F1 Score: 0.6006
  MCC: 0.5536
  AUC Score: 0.9126
  FNR: 0.0461, FDR: 0.0461, FPR: 0.3932, FoR: 0.3764
  Training Time: 2.1180 seconds
  Prediction Time: 0.5022 seconds
  Total Execution Time: 2.7050 seconds
  Confusion Matrix:
[[ 4

...


Training XGB...

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.83      0.80     14210
           1       0.98      1.00      0.99     13933
           2       0.88      0.68      0.77     13992
           3       0.74      0.91      0.82     14169
           4       1.00      1.00      1.00     14141
           5       0.85      0.78      0.81     14014
           6       0.94      0.89      0.91     14121
           7       0.95      1.00      0.98     14225

    accuracy                           0.89    112805
   macro avg       0.89      0.89      0.88    112805
weighted avg       0.89      0.89      0.88    112805


Evaluating XGB:
  Accuracy: 0.8859
  Precision: 0.8901
  Recall: 0.8859
  F1 Score: 0.8848
  MCC: 0.8706
  AUC Score: 0.9908
  FNR: 0.0139, FDR: 0.0139, FPR: 0.1143, FoR: 0.1098
  Training Time: 45.2616 seconds
  Prediction Time: 4.6992 seconds
  Total Execution Time: 50.0261 seconds
  Confusion Matrix:
[[


Training HGB...

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95     14210
           1       1.00      1.00      1.00     13933
           2       0.90      0.69      0.78     13992
           3       0.75      0.92      0.83     14169
           4       1.00      1.00      1.00     14141
           5       0.97      0.95      0.96     14014
           6       0.98      0.98      0.98     14121
           7       1.00      1.00      1.00     14225

    accuracy                           0.94    112805
   macro avg       0.94      0.94      0.94    112805
weighted avg       0.94      0.94      0.94    112805


Evaluating HGB:
  Accuracy: 0.9375
  Precision: 0.9419
  Recall: 0.9375
  F1 Score: 0.9368
  MCC: 0.9295
  AUC Score: 0.9955
  FNR: 0.0076, FDR: 0.0076, FPR: 0.0628, FoR: 0.0580
  Training Time: 77.8314 seconds
  Prediction Time: 33.0110 seconds
  Total Execution Time: 110.9017 seconds
  Confusion Matrix:



Training CatB...

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.73      0.72     14210
           1       0.94      0.99      0.96     13933
           2       0.88      0.63      0.73     13992
           3       0.72      0.91      0.80     14169
           4       0.99      1.00      0.99     14141
           5       0.77      0.70      0.73     14014
           6       0.92      0.84      0.88     14121
           7       0.91      0.99      0.95     14225

    accuracy                           0.85    112805
   macro avg       0.85      0.85      0.85    112805
weighted avg       0.85      0.85      0.85    112805


Evaluating CatB:
  Accuracy: 0.8496
  Precision: 0.8539
  Recall: 0.8496
  F1 Score: 0.8470
  MCC: 0.8294
  AUC Score: 0.9851
  FNR: 0.0183, FDR: 0.0182, FPR: 0.1506, FoR: 0.1460
  Training Time: 69.9025 seconds
  Prediction Time: 5.6521 seconds
  Total Execution Time: 75.6298 seconds
  Confusion Matrix:



Training LGB...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.061125 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9062
[LightGBM] [Info] Number of data points in the train set: 263211, number of used features: 39
[LightGBM] [Info] Start training from score -2.082771
[LightGBM] [Info] Start training from score -2.074360
[LightGBM] [Info] Start training from score -2.076145
[LightGBM] [Info] Start training from score -2.081522
[LightGBM] [Info] Start training from score -2.080669
[LightGBM] [Info] Start training from score -2.076812
[LightGBM] [Info] Start training from score -2.080061
[LightGBM] [Info] Start training from score -2.083229



Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.95      0.92     14210
           1       1.00      1.00      1.00     13933
           2       0.90      0.68      0.78     13992
           3       0.75      0.93      0.83     14169
           4       1.00      1.00      1.00     14141
           5       0.92      0.94      0.93     14014
           6       0.98      0.98      0.98     14121
           7       1.00      0.93      0.96     14225

    accuracy                           0.93    112805
   macro avg       0.93      0.93      0.93    112805
weighted avg       0.93      0.93      0.93    112805


Evaluating LGB:
  Accuracy: 0.9259
  Precision: 0.9315
  Recall: 0.9259
  F1 Score: 0.9253
  MCC: 0.9163
  AUC Score: 0.9951
  FNR: 0.0090, FDR: 0.0090, FPR: 0.0743, FoR: 0.0684
  Training Time: 46.2821 seconds
  Prediction Time: 17.6131 seconds
  Total Execution Time: 63.9537 seconds
  Confusion Matrix:
[[13527     5     

Summary of all models (accuracy, precision, recall and F1 correspond to Table II of the paper):

In [ ]:
summary = pd.DataFrame(results).T[['Accuracy', 'Precision', 'Recall', 'F1 Score', 'MCC', 'AUC Score', 'Total Execution Time']]
summary.astype(float).round(4)

      Accuracy  Precision  Recall  F1 Score     MCC  AUC Score  Total Execution Time
SGD     0.6264     0.6304  0.6264    0.6157  0.5766     0.9219               21.5977
LR      0.6525     0.6549  0.6525    0.6442  0.6051     0.9263              153.1744
NB      0.5098     0.6286  0.5098    0.4389  0.4898     0.8804                1.1899
LDA     0.6069     0.6231  0.6069    0.6006  0.5536     0.9126                2.7050
QDA     0.5652     0.6322  0.5652    0.5150  0.5349     0.9081                2.7906
AdaB    0.2815     0.4878  0.2815    0.2364  0.1881     0.8270              185.1729
XGB     0.8859     0.8901  0.8859    0.8848  0.8706     0.9908               50.0261
HGB     0.9375     0.9419  0.9375    0.9368  0.9295     0.9955              110.9017
CatB    0.8496     0.8539  0.8496    0.8470  0.8294     0.9851               75.6298
LGB     0.9259     0.9315  0.9259    0.9253  0.9163     0.9951               63.9537

## Figures

### ROC curves

In [ ]:
# --- Imports ---
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42  # TrueType fonts (no Type 3)
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['text.usetex'] = False

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# --- Create the plot ---
plt.close('all')
plt.figure(figsize=(12, 8))

for model_name, model in models.items():
    y_prob = results[model_name]['Probabilities']
    if y_prob is not None:
        fpr, tpr, _ = roc_curve(y_test_binarized.ravel(), y_prob.ravel())
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=3, label=f'{model_name} (AUC = {roc_auc:.2f})')

# Plot settings
plt.plot([0, 1], [0, 1], color='navy', linestyle='--', linewidth=3)
plt.xlabel('False Positive Rate', fontsize=22)
plt.ylabel('True Positive Rate', fontsize=22)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.legend(loc='best', fontsize=20)
plt.grid(True)
plt.tight_layout()

# --- Save as high-res PNG (EDAS-safe) ---
out_path = "figures/AUC_ROC_Curve_plot.png"
plt.savefig(out_path, dpi=600, bbox_inches="tight")

# --- Show plot ---
plt.show()

### Precision-recall curves

In [ ]:
# --- Imports ---
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42  # TrueType (avoids Type 3 in PDF)
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['text.usetex'] = False

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

# --- Create the plot ---
plt.close('all')
plt.figure(figsize=(12, 8))

for model_name, model in models.items():
    y_prob = results[model_name]['Probabilities']
    if y_prob is not None:
        precision, recall, _ = precision_recall_curve(
            y_test_binarized.ravel(), y_prob.ravel()
        )
        avg_precision = average_precision_score(
            y_test_binarized, y_prob, average='weighted'
        )
        plt.plot(recall, precision, lw=3,
                 label=f'{model_name} (AP = {avg_precision:.2f})')

# Labels & Styling
plt.xlabel('Recall', fontsize=22)
plt.ylabel('Precision', fontsize=22)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.legend(loc='best', fontsize=20)
plt.grid(True)
plt.tight_layout()

# --- Save as high-res PNG ---
out_path = "figures/precision_recall_plot.png"
plt.savefig(out_path, dpi=600, bbox_inches="tight")

# --- Show plot ---
plt.show()

### Error metrics (Fig. 2 of the paper)

In [ ]:
import matplotlib as mpl
import pandas as pd
import matplotlib.pyplot as plt

# Fix font type to avoid Type 3 fonts (for PDFs; harmless for PNGs)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['text.usetex'] = False

# --- Data Prep ---
metrics = ['FNR', 'FDR', 'FPR', 'FoR']
metric_values = []

for model_name in models:
    metric_values.append([
        results[model_name]['FNR'],
        results[model_name]['FDR'],
        results[model_name]['FPR'],
        results[model_name]['FoR']
    ])

error_metrics_df = pd.DataFrame(metric_values, columns=metrics, index=models.keys())

# --- Plot ---
plt.close('all')
error_metrics_df.plot(kind='bar', figsize=(12, 8), width=0.6,
                      color=['olive', 'blue', 'green', 'orange'])
plt.xticks(rotation=45, fontsize=22)
plt.yticks(fontsize=20)
plt.legend(loc='best', fontsize=20)
plt.grid(True)
plt.tight_layout()

# --- Save as high-res PNG ---
out_path = "figures/Error_plot.png"
plt.savefig(out_path, dpi=600, bbox_inches="tight")

# --- Show ---
plt.show()

### Execution time (Fig. 3 of the paper)

The execution times below are the values measured in the run reported in the paper.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

# Fix font type to avoid Type 3 fonts (for PDFs; harmless for PNGs)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['text.usetex'] = False

# Total Execution Times (in seconds)
execution_times = [
    21.1932, 93.2821, 1.5273, 2.1923, 5.0213,
    160.3051, 49.6226, 104.7527, 68.7278, 64.9030
]

# Model Names
models = [
    "SGD", "LR", "NB", "LDA", "QDA", "AdaB",
    "XGB", "HGB", "CatB", "LGB"
]

# Create x positions for each model
x = np.linspace(0, 10 * np.pi, len(execution_times))  # Positioning for models

# Plot
plt.close('all')
plt.figure(figsize=(12, 8))

plt.plot(x, execution_times, marker='o', color='darkblue',
         linewidth=4, markersize=10)
plt.fill_between(x, 0, execution_times, color='lightblue', alpha=0.3)

plt.ylim(0, 170)
plt.yticks(np.arange(0, 161, 20), fontsize=20)
plt.ylabel('Execution Time (seconds)', fontsize=22)
plt.xticks(x, models, rotation=45, fontsize=22)
plt.grid(True)
plt.tight_layout()

# --- Save as high-res PNG ---
out_path = "figures/execution_times_plot.png"
plt.savefig(out_path, dpi=600, bbox_inches="tight")

# Show plot
plt.show()

## Additional analysis

### Confusion matrix of a single model

Re-trains one model (here LightGBM) and plots its confusion matrix with class names. Note: this cell overwrites `models` and `results`.

In [ ]:
# Initialize Label Encoder
label_encoder = LabelEncoder()

# Encode target labels
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Binarize labels for multi-class AUC and PR Curve computation
n_classes = len(set(y_train_encoded))
y_test_binarized = label_binarize(y_test_encoded, classes=np.arange(n_classes))


# Correct SGDClassifier initialization
models = {
#   #  'Gradient Boosting': GradientBoostingClassifier(random_state=42),
#     'SGD': SGDClassifier(loss='log_loss', random_state=42),  # Corrected loss to 'log_loss'
#     #'Perceptron': Perceptron(random_state=42),
#     #'Ridge Classifier': RidgeClassifier(),
#     'LR': LogisticRegression(max_iter=500, random_state=42),
#     #'Decision Tree': DecisionTreeClassifier(random_state=42),
#     'NB': GaussianNB(),
#     'LDA': LinearDiscriminantAnalysis(),
#     'QDA': QuadraticDiscriminantAnalysis(),
#     'AdaB': AdaBoostClassifier(n_estimators=100, random_state=42),
#     'XGB': XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
#    'HGB': HistGradientBoostingClassifier(random_state=42),
#     'CatB': CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, verbose=0, random_state=42),
     'LGB': LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
#     #'GradientBoost': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
#     #'Extra Trees Classifier': ExtraTreesClassifier(n_estimators=100, random_state=42),
 }


# Store metrics for all models
results = {}

# Initialize plot for AUC-ROC curves
plt.figure(figsize=(10, 8))

# Train and evaluate all models
for model_name, model in models.items():
    print(f"\nTraining {model_name}...")

    # Track execution times
    start_total_time = time.time()
    start_train_time = time.time()

    # Train model
    model.fit(X_train, y_train_encoded)

    end_train_time = time.time()

    # Track prediction time
    start_pred_time = time.time()

    # Make predictions
    y_pred = model.predict(X_test)

    # Get probability predictions for Precision-Recall curve and AUC score
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)
        auc_score = roc_auc_score(y_test_binarized, y_prob, multi_class='ovr')
    else:
        # For models like Perceptron, RidgeClassifier, and SGD without predict_proba, we can't compute AUC.
        y_prob = None
        auc_score = np.nan

    end_pred_time = time.time()

    # Confusion matrix
    conf_matrix = confusion_matrix(y_test_encoded, y_pred)

    # Track total execution time
    end_total_time = time.time()

    # Convert label_encoder.classes_ to a list of strings
    target_names = [str(cls) for cls in label_encoder.classes_]


    # Store results
    results[model_name] = {

        'Confusion Matrix': conf_matrix,
       }

    print(f"\nEvaluating {model_name}:")
    print("  Confusion Matrix:")
    print(conf_matrix)
    print('-' * 50)


    # Plot Confusion Matrix with Class Names and Increased Text Size
    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(y_test_encoded, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", annot_kws={"size": 18},
            xticklabels=label_encoder.classes_,  # Use label_encoder.classes_ for x-axis labels
            yticklabels=label_encoder.classes_) # Use label_encoder.classes_ for y-axis labels
    plt.xlabel("Predicted Labels", fontsize=24)
    plt.ylabel("True Labels", fontsize=24)
    plt.title(f"{model_name}", fontsize=22)
    plt.xticks(rotation=45, ha='right', fontsize=12)  # Rotate x-axis labels for better readability
    plt.yticks(fontsize=12)  # Increase y-axis label font size
    plt.tight_layout()  # Adjust layout to prevent overlapping elements
    plt.show()  # Display the confusion matrix


Training LGB...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.092747 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9062
[LightGBM] [Info] Number of data points in the train set: 263211, number of used features: 39
[LightGBM] [Info] Start training from score -2.082771
[LightGBM] [Info] Start training from score -2.074360
[LightGBM] [Info] Start training from score -2.076145
[LightGBM] [Info] Start training from score -2.081522
[LightGBM] [Info] Start training from score -2.080669
[LightGBM] [Info] Start training from score -2.076812
[LightGBM] [Info] Start training from score -2.080061
[LightGBM] [Info] Start training from score -2.083229



Evaluating LGB:
  Confusion Matrix:
[[13527     5     0     0     0   525   152     1]
 [    0 13931     0     0     0     2     0     0]
 [    0     0  9544  4445     3     0     0     0]
 [    0     0  1035 13131     3     0     0     0]
 [    0     0     0     0 14141     0     0     0]
 [  770     0     0     0     0 13165    74     5]
 [  226     1     0     0     0    61 13830     3]
 [  548     0     0     0     0   499     1 13177]]
--------------------------------------------------


In [ ]:
# Define class names mapping
class_names = {
    0: "BENIGN",
    1: "Brute_Force",
    2: "DDOS",
    3: "DOS",
    4: "Mirai",
    5: "Recon",
    6: "Spoofing",
    7: "Web-based"
}

# Plot Confusion Matrix with Class Names and Increased Text Size
plt.figure(figsize=(12, 10))  # Increase figure size to better accommodate labels
cm = confusion_matrix(y_test_encoded, y_pred)

# Use the class names mapping for labels
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", annot_kws={"size": 22},
            xticklabels=[class_names[i] for i in range(len(class_names))],  # Use class names
            yticklabels=[class_names[i] for i in range(len(class_names))],  # Use class names
            cbar=False)  # Remove color bar for cleaner look
plt.xlabel("Predicted Labels", fontsize=28)
plt.ylabel("True Labels", fontsize=28)
plt.title(f"{model_name}", fontsize=40)
plt.xticks(rotation=45, ha='right', fontsize=24)  # Rotate x-axis labels for better readability
plt.yticks(rotation=0, fontsize=24)  # No rotation on y-axis, adjust font size

plt.tight_layout()  # Adjust layout to prevent overlapping elements
# Grid and legend

plt.show()  # Display the confusion matrix

### Feature importance (LightGBM)

In [ ]:
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Train
lgb_model = LGBMClassifier(
    n_estimators=100, learning_rate=0.01, max_depth=12, min_child_weight=5,
    subsample=0.85, colsample_bytree=0.75, gamma=0.3, reg_alpha=0.2, reg_lambda=0.5,
    scale_pos_weight=1, objective='multiclass', metric='multi_logloss', random_state=42
)
lgb_model.fit(X_train, y_train)

# --- Get aligned feature names safely ---
feature_names = getattr(lgb_model, "feature_name_", None)
if feature_names is None or len(feature_names) != X_train.shape[1]:
    feature_names = list(X_train.columns)  # fallback

# --- Importances ---
importance = np.array(lgb_model.feature_importances_, dtype=float)

# Avoid div-by-zero if all are zero
max_imp = importance.max() if importance.size and importance.max() > 0 else 1.0
importance_scaled = importance / max_imp

# Build sorted DataFrame
importance_df = (
    pd.Series(importance_scaled, index=feature_names, name="Importance")
      .sort_values(ascending=False)
      .reset_index()
      .rename(columns={"index": "Feature"})
)

# --- Plot ---
plt.figure(figsize=(30, 20))
sns.barplot(x="Importance", y="Feature", data=importance_df, palette="grey")

for i, val in enumerate(importance_df["Importance"]):
    plt.text(val + 0.005, i, f"{val:.3f}", ha="left", va="center", fontsize=28)

plt.xlabel("Importance", fontsize=50)
plt.ylabel("")
plt.yticks(fontsize=30)
plt.xticks(fontsize=30)
plt.tight_layout()

# --- Save as high-quality PNG ---
plt.savefig(
    "figures/Features_IMP.png",
    format="png",
    dpi=600,               # high resolution
    bbox_inches="tight",
    pad_inches=0.1,
    facecolor="white"
)
plt.show()

### Radar chart of the evaluation metrics

Values from Table II of the paper, with MCC and AUC from the same run, min-max normalised per metric.

In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

# Fix font type to avoid Type 3 fonts (for PDFs; harmless for PNGs)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['text.usetex'] = False

plt.style.use('default')

# Data (added MCC values)
data = {
    'Model': ['SGD', 'LR', 'NB', 'LDA', 'QDA', 'AdaB', 'XGB', 'HGB', 'CatB', 'LGB'],
    'Accuracy': [0.6264, 0.6529, 0.5098, 0.6069, 0.5652, 0.2815, 0.8859, 0.9375, 0.8496, 0.9259],
    'Precision': [0.6304, 0.6551, 0.6286, 0.6231, 0.6322, 0.4878, 0.8901, 0.9419, 0.8539, 0.9315],
    'Recall': [0.6264, 0.6529, 0.5098, 0.6069, 0.5652, 0.2815, 0.8859, 0.9375, 0.8496, 0.9259],
    'F1 Score': [0.6157, 0.6445, 0.4389, 0.6006, 0.5150, 0.2364, 0.8848, 0.9368, 0.8470, 0.9253],
    'MCC': [0.5766, 0.6055, 0.4898, 0.5536, 0.5349, 0.1881, 0.8706, 0.9295, 0.8294, 0.9163],
    'AUC Score': [0.9219, 0.9264, 0.8804, 0.9126, 0.9081, 0.8270, 0.9908, 0.9955, 0.9851, 0.9951]
}

df = pd.DataFrame(data)

# Normalize metrics for plotting
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'MCC', 'AUC Score']
df_norm = df.copy()
df_norm[metrics] = (df[metrics] - df[metrics].min()) / (df[metrics].max() - df[metrics].min())

# Radar chart setup
labels = metrics
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

fig = plt.figure(figsize=(12, 12), facecolor='white')
ax = fig.add_subplot(111, polar=True)
ax.set_facecolor('white')

# Plot each model
for i, row in df_norm.iterrows():
    values = row[metrics].tolist()
    values += values[:1]
    ax.plot(angles, values, label=row['Model'], linewidth=4)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=24)
ax.set_rgrids([0.2, 0.4, 0.6, 0.8, 1.0], angle=45, fontsize=20)
ax.set_ylim(0, 1)

ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1.08), fontsize=20)

plt.tight_layout()

# --- Save as high-res PNG ---
out_path = "figures/Radar.png"
plt.savefig(out_path, dpi=600, bbox_inches="tight")